# Final audio pipeline

Chạy cell dưới sau khi cài package:
`!pip -q install faster-whisper librosa soundfile tqdm`


In [ ]:
from pathlib import Path
import json
import zipfile
import traceback

import numpy as np
import librosa
import soundfile as sf
from tqdm.auto import tqdm

# =========================
# CẤU HÌNH
# =========================
# Nếu biết chắc path input, điền vào đây.
# Ví dụ:
# MANUAL_INPUT_PATH = "/kaggle/input/datasets/nguyendangvianh/my-audio-data-zip/sound_"
MANUAL_INPUT_PATH = None

OUTPUT_DIR = Path("/kaggle/working/audio_pipeline_final")
TARGET_SR = 16000
LANGUAGE = "vi"
RUN_ASR = True
ASR_MODEL = "tiny"      # tiny an toàn hơn small trên Kaggle CPU
BEAM_SIZE = 1            # nhẹ hơn beam_size=5
MIN_SEG_DUR = 1.0
MAX_SEG_DUR = 15.0
MAX_FILES = None         # ví dụ: 2 để test nhanh, None để chạy tất cả

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".wma"}

RAW_DIR = OUTPUT_DIR / "raw"
FULL_WAV_DIR = OUTPUT_DIR / "full_wavs"
SEGMENT_DIR = OUTPUT_DIR / "segments"
META_DIR = OUTPUT_DIR / "metadata"
for d in [RAW_DIR, FULL_WAV_DIR, SEGMENT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# IMPORT ASR NẾU CẦN
# =========================
WhisperModel = None
if RUN_ASR:
    try:
        from faster_whisper import WhisperModel
    except Exception as e:
        raise ImportError(
            "RUN_ASR=True nhưng chưa cài faster-whisper. Chạy cell này trước:\n"
            "!pip -q install faster-whisper librosa soundfile tqdm"
        ) from e


def load_whisper_model(model_name: str = "tiny"):
    device = "cuda" if Path("/dev/nvidia0").exists() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    print(f"[INFO] ASR device={device}, compute_type={compute_type}, model={model_name}")
    return WhisperModel(model_name, device=device, compute_type=compute_type)


def list_kaggle_input():
    base = Path("/kaggle/input")
    print("=== INPUT TÌM THẤY ===")
    if not base.exists():
        print("Không có /kaggle/input")
        return
    for p in sorted(base.rglob("*")):
        if p.is_file() and (p.suffix.lower() in AUDIO_EXTS or p.suffix.lower() == ".zip"):
            print(p)


def detect_input_path() -> Path:
    if MANUAL_INPUT_PATH:
        p = Path(MANUAL_INPUT_PATH)
        print(f"[INFO] Dùng path chỉ định tay: {p}")
        return p

    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("Không tìm thấy /kaggle/input")

    zip_files = sorted([p for p in base.rglob("*.zip") if p.is_file()])
    if zip_files:
        print(f"[INFO] Tự động chọn file zip: {zip_files[0]}")
        return zip_files[0]

    audio_files = sorted([p for p in base.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS])
    if audio_files:
        print(f"[INFO] Tự động chọn thư mục audio: {audio_files[0].parent}")
        return audio_files[0].parent

    dirs = sorted([p for p in base.iterdir() if p.is_dir()])
    if dirs:
        print(f"[INFO] Fallback chọn thư mục: {dirs[0]}")
        return dirs[0]

    raise FileNotFoundError("Không tìm thấy zip hay file audio nào trong /kaggle/input")


def prepare_input(input_path: Path, raw_dir: Path) -> Path:
    if not input_path.exists():
        raise FileNotFoundError(f"Path không tồn tại: {input_path}")

    if input_path.is_dir():
        return input_path

    if input_path.is_file() and zipfile.is_zipfile(input_path):
        with zipfile.ZipFile(input_path, "r") as zf:
            zf.extractall(raw_dir)
        return raw_dir

    raise ValueError(f"INPUT_PATH không phải thư mục hoặc file zip hợp lệ: {input_path}")


def find_audio_files(root: Path):
    files = sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS])
    if MAX_FILES is not None:
        files = files[:MAX_FILES]
    return files


def infer_speaker(audio_path: Path, root: Path) -> str:
    rel = audio_path.relative_to(root)
    if len(rel.parts) >= 2:
        return rel.parts[0]
    return "spk0"


def normalize_audio(infile: Path, outfile: Path, target_sr: int = 16000):
    y, sr = librosa.load(str(infile), sr=target_sr, mono=True)
    if y is None or len(y) == 0:
        raise ValueError(f"Audio rỗng: {infile}")

    peak = np.max(np.abs(y))
    if peak > 0:
        y = 0.98 * y / max(peak, 1e-8)

    outfile.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(outfile), y, target_sr)
    duration = len(y) / target_sr
    return y, target_sr, duration


def write_json(path: Path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def write_jsonl(path: Path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def run_pipeline():
    list_kaggle_input()

    input_path = detect_input_path()
    print(f"\n[INFO] INPUT_PATH: {input_path}")

    root = prepare_input(input_path, RAW_DIR)
    print(f"[INFO] Root xử lý: {root}")

    audio_files = find_audio_files(root)
    if not audio_files:
        raise RuntimeError(f"Không tìm thấy audio trong: {root}")

    print(f"[INFO] Số file audio: {len(audio_files)}")

    asr_model = load_whisper_model(ASR_MODEL) if RUN_ASR else None

    manifest = []
    errors = []
    speaker_to_id = {}

    for audio_path in tqdm(audio_files, desc="Processing"):
        try:
            speaker = infer_speaker(audio_path, root)
            if speaker not in speaker_to_id:
                speaker_to_id[speaker] = len(speaker_to_id)

            rel = audio_path.relative_to(root)
            full_wav_path = FULL_WAV_DIR / rel.with_suffix(".wav")
            y, sr, full_duration = normalize_audio(audio_path, full_wav_path, TARGET_SR)

            if not RUN_ASR:
                manifest.append({
                    "id": f"{speaker}_{full_wav_path.stem}",
                    "speaker": speaker,
                    "speaker_id": speaker_to_id[speaker],
                    "audio_filepath": str(full_wav_path),
                    "duration": round(float(full_duration), 4),
                    "sampling_rate": sr,
                    "language": LANGUAGE,
                    "text": "",
                    "source_audio": str(audio_path),
                    "start": 0.0,
                    "end": round(float(full_duration), 4),
                })
                continue

            segments, info = asr_model.transcribe(
                str(full_wav_path),
                language=LANGUAGE,
                vad_filter=True,
                beam_size=BEAM_SIZE,
            )
            detected_lang = getattr(info, "language", LANGUAGE)

            for idx, seg in enumerate(segments):
                start = max(0.0, float(seg.start))
                end = min(float(seg.end), full_duration)
                text = (seg.text or "").strip()
                dur = end - start

                if dur < MIN_SEG_DUR or dur > MAX_SEG_DUR or not text:
                    continue

                s0 = int(start * sr)
                s1 = int(end * sr)
                clip = y[s0:s1]
                if len(clip) == 0:
                    continue

                seg_name = f"{full_wav_path.stem}_seg{idx:04d}.wav"
                seg_path = SEGMENT_DIR / speaker / seg_name
                seg_path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(str(seg_path), clip, sr)

                manifest.append({
                    "id": f"{speaker}_{full_wav_path.stem}_seg{idx:04d}",
                    "speaker": speaker,
                    "speaker_id": speaker_to_id[speaker],
                    "audio_filepath": str(seg_path),
                    "duration": round(float(dur), 4),
                    "sampling_rate": sr,
                    "language": detected_lang,
                    "text": text,
                    "source_audio": str(audio_path),
                    "normalized_audio": str(full_wav_path),
                    "start": round(start, 4),
                    "end": round(end, 4),
                })

        except KeyboardInterrupt:
            print("\n[INFO] Đã dừng tay. Sẽ lưu phần đã chạy xong.")
            break
        except Exception as e:
            errors.append({
                "file": str(audio_path),
                "error": str(e),
                "traceback": traceback.format_exc(limit=1),
            })

    write_json(META_DIR / "speakers.json", speaker_to_id)
    write_json(META_DIR / "manifest.json", manifest)
    write_jsonl(META_DIR / "manifest.jsonl", manifest)
    write_json(META_DIR / "errors.json", errors)

    stats = {
        "input_path": str(input_path),
        "root_used": str(root),
        "num_input_files": len(audio_files),
        "num_manifest_rows": len(manifest),
        "num_speakers": len(speaker_to_id),
        "num_errors": len(errors),
        "target_sr": TARGET_SR,
        "run_asr": RUN_ASR,
        "language": LANGUAGE,
        "asr_model": ASR_MODEL if RUN_ASR else None,
    }
    write_json(META_DIR / "stats.json", stats)

    print("\n=== KẾT QUẢ ===")
    print("Manifest JSON :", META_DIR / "manifest.json")
    print("Manifest JSONL:", META_DIR / "manifest.jsonl")
    print("Speakers      :", META_DIR / "speakers.json")
    print("Errors        :", META_DIR / "errors.json")
    print(json.dumps(stats, ensure_ascii=False, indent=2))

    if manifest:
        print("\n=== 5 DÒNG ĐẦU MANIFEST ===")
        for i, row in enumerate(manifest[:5], start=1):
            print(f"\n--- Row {i} ---")
            print(json.dumps(row, ensure_ascii=False, indent=2))
    else:
        print("\n[INFO] Manifest đang rỗng.")


if __name__ == "__main__":
    run_pipeline()
